# 📊 Análisis de Red de Tiendas — RetailNow
**Herramientas:** Python · Pandas · NumPy  
**Objetivo:** Analizar ventas, inventarios y satisfacción del cliente para optimizar el rendimiento de las sucursales.


## 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import os

## 2. Cargar y limpiar datos

Cargamos los tres CSV en DataFrames independientes y eliminamos filas con valores nulos con `dropna()`.

In [2]:
df_ventas       = pd.read_csv("workspace/ventas.csv")
df_inventarios  = pd.read_csv("workspace/inventarios.csv")
df_satisfaccion = pd.read_csv("workspace/satisfaccion.csv")

df_ventas       = df_ventas.dropna()
df_inventarios  = df_inventarios.dropna()
df_satisfaccion = df_satisfaccion.dropna()

# Verificamos que no quedan nulos
print("Nulos en ventas:      ", df_ventas.isnull().sum().sum())
print("Nulos en inventarios: ", df_inventarios.isnull().sum().sum())
print("Nulos en satisfaccion:", df_satisfaccion.isnull().sum().sum())

# Estructura, shape y tipos de los DataFrames tras la limpieza
for nombre, df in [("df_ventas", df_ventas), ("df_inventarios", df_inventarios), ("df_satisfaccion", df_satisfaccion)]:
    print(f"\n--- {nombre} {df.shape} ---")
    print(df.head())
    df.info()

Nulos en ventas:       0
Nulos en inventarios:  0
Nulos en satisfaccion: 0

--- df_ventas (10, 5) ---
   ID_Tienda    Producto  Cantidad_Vendida  Precio_Unitario Fecha_Venta
0          1  Producto A                20              100  2023-01-05
1          1  Producto B                15              200  2023-01-06
2          2  Producto A                30              100  2023-01-07
3          2  Producto C                25              300  2023-01-08
4          3  Producto A                10              100  2023-01-09
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ID_Tienda         10 non-null     int64
 1   Producto          10 non-null     str  
 2   Cantidad_Vendida  10 non-null     int64
 3   Precio_Unitario   10 non-null     int64
 4   Fecha_Venta       10 non-null     str  
dtypes: int64(3), str(2)
memory usage: 532.0 bytes

--- df_in

## 3. Exploración de ventas

Calculamos ventas totales por tienda y por producto, los ingresos por tienda y un resumen estadístico. Como el CSV no tiene columna `Categoria`, agrupamos por `Producto` como criterio equivalente.

In [3]:
# Ventas totales por tienda y por producto — guardadas en variables para reutilización
ventas_totales_tienda   = df_ventas.groupby("ID_Tienda")["Cantidad_Vendida"].sum()
ventas_totales_producto = df_ventas.groupby("Producto")["Cantidad_Vendida"].sum()

print("Ventas totales por tienda:")
print(ventas_totales_tienda)

print("\nVentas totales por producto:")
print(ventas_totales_producto)

Ventas totales por tienda:
ID_Tienda
1    35
2    55
3    50
4    60
5    50
Name: Cantidad_Vendida, dtype: int64

Ventas totales por producto:
Producto
Producto A    85
Producto B    75
Producto C    90
Name: Cantidad_Vendida, dtype: int64


In [4]:
# Ingresos totales por tienda (Cantidad_Vendida × Precio_Unitario)
df_ventas["Ingresos"] = df_ventas["Cantidad_Vendida"] * df_ventas["Precio_Unitario"]

print("Ingresos totales por tienda:")
print(df_ventas.groupby("ID_Tienda")["Ingresos"].sum())

Ingresos totales por tienda:
ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Ingresos, dtype: int64


In [5]:
# Resumen estadístico de las ventas
print(df_ventas["Cantidad_Vendida"].describe())

count    10.000000
mean     25.000000
std       9.128709
min      10.000000
25%      20.000000
50%      25.000000
75%      30.000000
max      40.000000
Name: Cantidad_Vendida, dtype: float64


In [6]:
# Promedio de ventas por tienda y producto (equivalente a tienda + categoría)
print("Promedio de ventas por tienda y producto:")
print(df_ventas.groupby(["ID_Tienda", "Producto"])["Cantidad_Vendida"].mean())

Promedio de ventas por tienda y producto:
ID_Tienda  Producto  
1          Producto A    20.0
           Producto B    15.0
2          Producto A    30.0
           Producto C    25.0
3          Producto A    10.0
           Producto B    40.0
4          Producto A    25.0
           Producto C    35.0
5          Producto B    20.0
           Producto C    30.0
Name: Cantidad_Vendida, dtype: float64


## 4. Análisis de inventarios

Calculamos la rotación de inventarios: `Cantidad_Vendida / Stock_Disponible`.  
Un valor menor al 10% indica un nivel crítico que requiere atención.

In [7]:
# Ventas totales por tienda y producto para cruzar con inventarios
ventas_agrupadas = df_ventas.groupby(["ID_Tienda", "Producto"])["Cantidad_Vendida"].sum()

# Añadimos las ventas al DataFrame de inventarios
df_inventarios = df_inventarios.join(ventas_agrupadas, on=["ID_Tienda", "Producto"])

# fillna(0) para evitar NaN si alguna combinación tienda-producto no tiene ventas
df_inventarios["Cantidad_Vendida"] = df_inventarios["Cantidad_Vendida"].fillna(0)

# Rotación = proporción vendida respecto al stock. Crítico si < 10% (pocas ventas vs stock)
df_inventarios["Rotacion"] = df_inventarios["Cantidad_Vendida"] / df_inventarios["Stock_Disponible"]

print("Rotación de inventarios por tienda y producto:")
print(df_inventarios[["ID_Tienda", "Producto", "Stock_Disponible", "Cantidad_Vendida", "Rotacion"]])

# Rotación media agregada por tienda
print("\nRotación media por tienda:")
print(df_inventarios.groupby("ID_Tienda")["Rotacion"].mean())

Rotación de inventarios por tienda y producto:
   ID_Tienda    Producto  Stock_Disponible  Cantidad_Vendida  Rotacion
0          1  Producto A                50                20  0.400000
1          1  Producto B                40                15  0.375000
2          2  Producto A                60                30  0.500000
3          2  Producto C                45                25  0.555556
4          3  Producto A                30                10  0.333333
5          3  Producto B                80                40  0.500000
6          4  Producto C                70                35  0.500000
7          4  Producto A                50                25  0.500000
8          5  Producto B                40                20  0.500000
9          5  Producto C                60                30  0.500000

Rotación media por tienda:
ID_Tienda
1    0.387500
2    0.527778
3    0.416667
4    0.500000
5    0.500000
Name: Rotacion, dtype: float64


In [8]:
# Tiendas con inventario crítico: menos del 10% del stock disponible ha sido vendido
criticos = df_inventarios[df_inventarios["Rotacion"] < 0.10]

if criticos.empty:
    print("No hay registros con inventario crítico (rotación < 10%). Todos los productos tienen ventas suficientes.")
else:
    print(f"Registros con inventario crítico (rotación < 10%): {len(criticos)}")
    print(criticos[["ID_Tienda", "Producto", "Stock_Disponible", "Cantidad_Vendida", "Rotacion"]])
    print("\nProductos críticos por tienda:")
    print(criticos.groupby("ID_Tienda")["Producto"].count())

No hay registros con inventario crítico (rotación < 10%). Todos los productos tienen ventas suficientes.


## 5. Satisfacción del cliente

Identificamos las tiendas con satisfacción menor al 60% y planteamos recomendaciones de mejora.

In [9]:
print("Satisfacción por tienda:")
print(df_satisfaccion[["ID_Tienda", "Satisfacción_Promedio"]])

Satisfacción por tienda:
   ID_Tienda  Satisfacción_Promedio
0          1                     85
1          2                     90
2          3                     70
3          4                     65
4          5                     55


In [10]:
# Filtrar tiendas con baja satisfacción (< 60%)
baja_satisfaccion = df_satisfaccion[df_satisfaccion["Satisfacción_Promedio"] < 60]

print(f"Tiendas con satisfacción < 60%: {len(baja_satisfaccion)}")
print(baja_satisfaccion[["ID_Tienda", "Satisfacción_Promedio"]])

Tiendas con satisfacción < 60%: 1
   ID_Tienda  Satisfacción_Promedio
4          5                     55


In [11]:
# Recomendaciones para tiendas con baja satisfacción
for _, fila in baja_satisfaccion.iterrows():
    print(f"\n⚠️  Tienda {fila['ID_Tienda']} — Satisfacción: {fila['Satisfacción_Promedio']}%")
    print("   → Revisar atención al cliente y procesos de servicio.")
    print("   → Analizar causas raíz con los comentarios de clientes.")
    print("   → Considerar formación adicional para el personal.")


⚠️  Tienda 5 — Satisfacción: 55%
   → Revisar atención al cliente y procesos de servicio.
   → Analizar causas raíz con los comentarios de clientes.
   → Considerar formación adicional para el personal.


In [12]:
# Relación entre satisfacción y ventas por tienda
satisfaccion_ventas = pd.merge(df_satisfaccion[["ID_Tienda", "Satisfacción_Promedio"]], ventas_totales_tienda, on="ID_Tienda")
print("Satisfacción y ventas por tienda:")
print(satisfaccion_ventas)

# Correlación entre satisfacción y ventas
correlacion = satisfaccion_ventas["Satisfacción_Promedio"].corr(satisfaccion_ventas["Cantidad_Vendida"])
print(f"\nCorrelación satisfacción-ventas: {correlacion:.2f}")

# Interpretación ejecutiva
if correlacion > 0.5:
    interpretacion = "positiva fuerte: tiendas con mayor satisfacción tienden a vender más."
elif correlacion > 0:
    interpretacion = "positiva débil: hay cierta tendencia a vender más en tiendas con mejor satisfacción."
elif correlacion > -0.5:
    interpretacion = "negativa débil: la relación entre satisfacción y ventas no es clara."
else:
    interpretacion = "negativa fuerte: tiendas con mayor satisfacción tienden a vender menos."
print(f"Interpretación: correlación {interpretacion}")

Satisfacción y ventas por tienda:
   ID_Tienda  Satisfacción_Promedio  Cantidad_Vendida
0          1                     85                35
1          2                     90                55
2          3                     70                50
3          4                     65                60
4          5                     55                50

Correlación satisfacción-ventas: -0.32
Interpretación: correlación negativa débil: la relación entre satisfacción y ventas no es clara.


## 6. Operaciones con NumPy

Usamos NumPy para calcular estadísticas sobre las ventas totales y simular proyecciones futuras.

In [13]:
# Estadísticas calculadas sobre ventas totales agregadas por tienda
# (cada valor del array = suma de todas las ventas de una tienda; no por transacción individual)
array_ventas = ventas_totales_tienda.to_numpy()

mediana  = np.median(array_ventas)
desv_std = np.std(array_ventas)

print(f"Mediana de ventas totales:      {mediana}")
print(f"Desviación estándar de ventas:  {desv_std:.2f}")

Mediana de ventas totales:      50.0
Desviación estándar de ventas:  8.37


In [14]:
# Parámetros configurables para facilitar experimentos con distintos escenarios
N_MESES     = 12
CRECIMIENTO = 1.05  # 5% de crecimiento estimado sobre la media histórica

np.random.seed(42)

proyecciones = np.random.normal(
    loc=np.mean(array_ventas) * CRECIMIENTO,  # media esperada = media histórica + crecimiento
    scale=np.std(array_ventas),               # variabilidad igual a la desviación estándar actual
    size=(N_MESES, len(array_ventas))
)

# Recortamos valores negativos que pudieran surgir de la distribución normal
proyecciones = np.clip(proyecciones, 0, None)

# DataFrame con índices de meses y columnas de tiendas
meses = [f"Mes_{i+1}" for i in range(N_MESES)]
tiendas = df_ventas["ID_Tienda"].unique()
df_proyecciones = pd.DataFrame(proyecciones.round(1), index=meses, columns=[f"Tienda_{t}" for t in sorted(tiendas)])

print("Proyecciones de ventas futuras:")
print(df_proyecciones)

# Resumen ejecutivo: media proyectada por tienda
print("\nMedia proyectada por tienda:")
print(df_proyecciones.mean().round(1))

# Guardamos en CSV para uso posterior
df_proyecciones.to_csv(os.path.abspath("proyecciones_ventas.csv"))
print("\nProyecciones guardadas en proyecciones_ventas.csv")

Proyecciones de ventas futuras:
        Tienda_1  Tienda_2  Tienda_3  Tienda_4  Tienda_5
Mes_1       56.7      51.3      57.9      65.2      50.5
Mes_2       50.5      65.7      58.9      48.6      57.0
Mes_3       48.6      48.6      54.5      36.5      38.1
Mes_4       47.8      44.0      55.1      44.9      40.7
Mes_5       64.8      50.6      53.1      40.6      47.9
Mes_6       53.4      42.9      55.6      47.5      50.1
Mes_7       47.5      68.0      52.4      43.7      59.4
Mes_8       42.3      54.2      36.1      41.4      54.1
Mes_9       58.7      53.9      51.5      50.0      40.1
Mes_10      46.5      48.6      61.3      55.4      37.7
Mes_11      55.2      49.3      46.8      57.6      61.1
Mes_12      60.3      45.5      49.9      55.3      60.7

Media proyectada por tienda:
Tienda_1    52.7
Tienda_2    51.9
Tienda_3    52.8
Tienda_4    48.9
Tienda_5    49.8
dtype: float64

Proyecciones guardadas en proyecciones_ventas.csv


In [15]:
# Estadísticas sobre las proyecciones
print(f"Media proyectada:    {np.mean(proyecciones):.2f}")
print(f"Mediana proyectada:  {np.median(proyecciones):.2f}")
print(f"Desv. estándar:      {np.std(proyecciones):.2f}")
print(f"Mínimo proyectado:   {np.min(proyecciones):.2f}")
print(f"Máximo proyectado:   {np.max(proyecciones):.2f}")

Media proyectada:    51.21
Mediana proyectada:  50.58
Desv. estándar:      7.54
Mínimo proyectado:   36.10
Máximo proyectado:   68.00
